# Fundamentals 13 - Single AgenticSystem Completo

Un solo AgenticSystem con tools, agente, runtime, output final, lineage, environment y eval. El LM opcional explica, pero el camino determinista siempre corre.


In [ ]:
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=6, max_turns=6)
local_runtime = toolkit.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
system = toolkit.AgenticSystem(model=lm_runtime.model_id or "local-python", region=lm_runtime.region_name or "local", runtime=lm_runtime)
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
toolkit.show({"runtime_auto": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


In [ ]:
@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@toolkit.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}


## Parametros de `RunPolicy`

`RunPolicy` declara c?mo debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | N?mero m?ximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | N?mero m?ximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | L?mite de tokens del modelo cuando el provider lo soporta. | ?til en providers LM; puede quedar `None` en `python-direct`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion autom?tica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | M?ximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


## 1) Sistema, contrato, policy y agente


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

single_system = toolkit.AgenticSystem(model="local-python", region="local", runtime=local_runtime)
policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
contract = toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic"), completion="when_required_tools_satisfied")
agent = single_system.agent(name="single_solver", instructions="Resuelve el problema estructurado.", tools=[solve_arithmetic], engine="python-direct", runtime=local_runtime, contract=contract, policy=policy)
explainer = system.agent(name="single_lm_explainer", instructions="Explica la soluci?n sin cambiar n?meros.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))
toolkit.show({"system": single_system.inspect(), "agent": agent.info(), "explainer": explainer.info()})


## 2) Run completo + final_answer + lineage


In [ ]:
solve = agent.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
explanation = None
explanation_result = None
if lm_available:
    explanation_result = explainer.run(str(solve.data))
    explanation = explanation_result.text
else:
    toolkit.show({"status": "skipped", "reason": lm_resolution["reason"]}, title="LM explainer saltado")

final = toolkit.final_answer(
    {"procedimiento": solve.data["procedure"], "resultado_final": solve.data["result"], "explicacion_lm": explanation},
    schema=toolkit.output_schema(fields=["procedimiento", "resultado_final", "explicacion_lm"]),
)
result = toolkit.compose_result(
    text="Single AgenticSystem completo ejecutado.",
    data=final,
    results=[solve, explanation_result],
    mode="single-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.single_agentic_system", question=USER_PROMPT, goal="Explicar el ciclo completo de un solo sistema.")
toolkit.human_result(result, title="Human result - Single AgenticSystem", pretty=PRETTY, show_lineage=True, lineage=lineage)

## 3) Environment y eval del mismo agente


In [ ]:
def transition_fn(row: dict, action: dict | None, info: dict) -> dict:
    out = agent.run({"tool": "solve_arithmetic", "input": {"numbers": row["numbers"]}}).data
    return {"result": out["result"], "expected": row["expected"], "ok": out["result"] == row["expected"]}

def reward_fn(state: dict) -> float:
    return 1.0 if state.get("ok") else 0.0

env_records = [{"numbers": NUMBERS, "expected": EXPECTED}]
env = toolkit.AgenticEnvironment(name="single_system_env", records=env_records, initial_memory={}, transition_fn=transition_fn, reward_fn=reward_fn)
env.reset()
_, reward, terminated, truncated, info = env.step()
report = toolkit.run_eval(agent, [{"id": "default", "input": {"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}}, "expected": {"result": EXPECTED}}])
toolkit.show({"env_summary": env.summary(), "step": info["transition"], "eval_report": report.to_dict()})


In [ ]:
toolkit.show({"notebook": "13_single_agentic_system_api.ipynb", "api_coverage": ["AgenticSystem", "tool", "agent", "runtime(provider='auto')", "final_answer", "compose_result", "RunResult.lineage", "AgenticEnvironment", "run_eval"]})


## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `AgenticSystem`: Ciclo completo single-agent.
- `Tool -> Skill -> Agent -> Environment -> Eval`: Ruta completa del framework.
- `final_answer / output_schema`: Salida estable de usuario.
- `LineageMemory`: Trazabilidad de ejecucion.

